# CPA attack on standard ASCON software implementation

This notebook performs the **CPA attack** on the previously acquired power traces.  
It loads the traceset produced by the acquisition notebook ( the h5 file (.h5)) and computes:

- **Correlation Power Analysis (CPA)** results  
- **Key rank** vs. number of traces  
- **Correlation trends** for each key byte, comparing correct vs. wrong key hypotheses  

These plots help evaluate the **difficulty of recovering each key byte** and visualize the leakage behavior across the trace window.

⚠️ **Important:**  
This notebook assumes that the **Power Trace Acquisition** notebook (`xheep_capture_ASCON.ipynb`) has already been executed and the traceset has been acquired.


In [ ]:
sbox_type   = "lut_ascon"   # do not modify this line
tested_sbox = sbox_type     # alias used later in the printout
n_trc       = 10_000        # total number of traces in the traceset

In [ ]:
import sys
import os
import time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import logging
import json
import h5py

## Project paths
This block robustly detects the project root (`DOJO_ROOT`) starting from either the script location (`__file__`) or the current working directory (for notebooks). From there it defines all relevant subdirectories (ASCON sources, SCA scripts, X-HEEP, traces, plots, cache), ensures output folders exist, and adds the local source paths to `sys.path` .

In [ ]:
def find_project_root(start: Path, markers=("fusesoc.conf", ".dojo_root")) -> Path:
    current = start
    while current != current.parent:
        if any((current / m).exists() for m in markers):
            return current
        current = current.parent

    raise RuntimeError(
        f"Could not find project root (looked for markers: {markers}). "
        "Please ensure you are inside the Side-Channel-Dojo repository."
    )

# In a script, __file__ exists; in a notebook it does not.
try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    SCRIPT_DIR = Path.cwd()

DOJO_ROOT = find_project_root(SCRIPT_DIR)

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------

ASCON_PY_DIR = DOJO_ROOT / "sw" / "ciphers" / "ASCON_init_python"
SCA_DIR      = DOJO_ROOT / "sw" / "sca_scripts"
HW_DIR       = DOJO_ROOT / "hw"

# Base dirs for ASCON SW SCA
BASE_PLOT_DIR  = DOJO_ROOT / "sw" / "sca_scripts" / "ASCON" / "sw" / "plot"
BASE_CACHE_DIR = DOJO_ROOT / "sw" / "sca_scripts" / "ASCON" / "sw" / "cache"

# Traceset (HDF5) for ASCON SW
TRACESET_DIR  = DOJO_ROOT / "sw" / "traceset" / "ASCON" / "sw"
TRACESET_FILE = TRACESET_DIR / f"ascon_opt32_{sbox_type}_{n_trc // 1000}k_tmp.h5"

# Per-S-box plot and cache dirs
PLOT_DIR       = BASE_PLOT_DIR / sbox_type
CPA_CACHE_FILE = BASE_CACHE_DIR / sbox_type / f"CPA_results_{sbox_type}.json"

# Ensure directories exist
BASE_PLOT_DIR.mkdir(parents=True, exist_ok=True)
BASE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)
CPA_CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)

# Import local modules
sys.path.insert(0, str(ASCON_PY_DIR))
sys.path.insert(0, str(SCA_DIR))

## Imports

# Configuration

In [ ]:
# ---------------------------------------------------------------------------
# Flow flags
# ---------------------------------------------------------------------------

traces_overlapped_plot   = True   # Plot overlapped power traces

# CPA / analysis cache control
load_attack_results      = False  # Load CPA cache if available
save_attack_results      = True   # Save CPA results to cache after the run

# Plot control
key_rank_plot            = True   # Plot PGE vs traces
traces_correlation_plot  = True   # Plot correlation vs traces

# Output control
save_plots               = True   # Save plots to disk
save_results             = True   # Save analysis results (JSON, etc.) to disk

# ---------------------------------------------------------------------------
# Configuration printout
# ---------------------------------------------------------------------------

def _yn(flag: bool) -> str:
    """Return 'yes' or 'no' for a boolean flag."""
    return "yes" if flag else "no"

print("\n================= CONFIGURATION =================")
print(f"DOJO_ROOT           : {DOJO_ROOT}")
print()
print("Target")
print(f"  Cipher                    : ASCON (standard permutation)")
print(f"  S-box implementation      : {tested_sbox}  [STANDARD ASCON S-box]")
print(f"  Notebook scope            : ASCON SW, standard S-box only")
print()
print("Paths")
print(f"  Traceset file             : {TRACESET_FILE}")
print(f"  Plot dir                  : {PLOT_DIR}")
print(f"  CPA cache file            : {CPA_CACHE_FILE}")
print()
print("Analysis configuration")
print(f"  Total traces (n_trc)      : {n_trc}")
print(f"  Save plots                : {_yn(save_plots)}")
print(f"  Save results              : {_yn(save_results)}")
print(f"  Load CPA results (cache)  : {_yn(load_attack_results)}")
print(f"  Save CPA results (cache)  : {_yn(save_attack_results)}")
print()
print(f"  Key rank plot             : {_yn(key_rank_plot)}")
print(f"  Correlation plot          : {_yn(traces_correlation_plot)}")
print("=================================================\n")

## SCA attack

## Load data to perform the attack

In [ ]:
try:
    with h5py.File(TRACESET_FILE, "r") as f_read_traces:
        # Basic sanity: check that required datasets exist
        if "traces" not in f_read_traces or "nonces" not in f_read_traces:
            raise KeyError(
                "HDF5 file is missing required datasets 'traces' and/or 'nonces'."
            )

        traces_ds = f_read_traces["traces"]
        nonces_ds = f_read_traces["nonces"]

        total_traces = traces_ds.shape[0]
        n_samples    = traces_ds.shape[1]

        # Load metadata attributes (if present)
        sampling_interval   = f_read_traces.attrs.get("sampling_interval", None)
        n_samples           = f_read_traces.attrs.get("n_samples", n_samples)
        key                 = f_read_traces.attrs.get("key_hex_lsb_first", None) # key is an hex string, lsb first
        iv                  = f_read_traces.attrs.get("iv_hex", None)

        # Use at most n_trc traces, but do not exceed what's in the file
        n_used = min(n_trc, total_traces)

        # Use slicing so data are actually loaded into RAM
        traces = traces_ds[:n_used]
        nonces = nonces_ds[:n_used]

    # Sanity check: traces and nonces should have the same number of rows
    if traces.shape[0] != nonces.shape[0]:
        raise ValueError(
            f"Number of traces ({traces.shape[0]}) and nonces ({nonces.shape[0]}) "
            "do not match. Check the traces file."
        )

    # Optional sanity checks vs metadata
    if n_trc != total_traces:
        print(
            f"[WARN] Wanted n_trc={n_trc} "
            f"differs from dataset length={total_traces}"
        )

    print(f"[INFO] Loaded {traces.shape[0]} traces from {TRACESET_FILE}")
    print(f"[INFO] Sampling interval      : {sampling_interval}")
    print(f"[INFO] Samples per trace      : {n_samples}")
    print(f"[INFO] Key (LSB-first, hex)   : {key}")
    print(f"[INFO] IV (hex)               : {iv}")

except FileNotFoundError:
    print(
        f"[ERROR] Traces file {TRACESET_FILE} not found. "
        "Please run the trace acquisition phase first."
    )
except Exception as e:
    print(f"[ERROR] Could not read traces file {TRACESET_FILE}: {e}")


## Attacking a single bit and plotting SNR and Correlation trend

In [ ]:

if cpa_phase_1_bit:
    print("Running CPA attack (this might take a while)...")
    # TODO: for the moment, the attack is performed only on the first half key register (x0).
    # Still needed to add an external loop over the necessary key bits to retrieve the
    # full key register x0. For each bit index, the leakage model is built and the CPA attack is performed.
    # The CPA attack is repeated with an incremental number of traces,
    # in order to see how the distance between the correlation value of 
    # the correct key guess and the others increases with the number of traces.
    corr_vs_traces = []
    state_register_index = 0
    bit_index = 60
    resolution = 5000
    k0 = key & 0xFFFFFFFFFFFFFFFF
    tic = time.perf_counter()
    # DEBUG
    key_0 = key & 0xFFFFFFFFFFFFFFFF
    key_0_j     = (key_0 >> (bit_index % 64)) & 1
    key_0_j19   = (key_0 >> ((bit_index + 19) % 64)) & 1
    key_0_j28   = (key_0 >> ((bit_index + 28) % 64)) & 1
    key_1 = (key >> 64) & 0xFFFFFFFFFFFFFFFF
    key_1_j     = (key_1 >> (bit_index % 64)) & 1
    key_1_j61   = (key_1 >> ((bit_index + 61) % 64)) & 1
    key_1_j39   = (key_1 >> ((bit_index + 39) % 64)) & 1
    for count in range(1, (traces.shape[0] // resolution) + 1):
        partial_traces = traces[:(count*resolution)]
        partial_nonces = nonces[:(count*resolution)]
        # Build the leakage model matrix for all the nonces
        H_matrix = np.empty((len(partial_nonces), 8), dtype=np.uint8)
        R_matrix = np.empty((8, partial_traces.shape[1]), dtype=np.float64)
        for n in range(len(partial_nonces)):
            nonce_MSB = partial_nonces[n][1]
            nonce_LSB = partial_nonces[n][0]
            leakage_model_i = ascon_leakage_model(initialization_vector, nonce_MSB, nonce_LSB, state_register_index, bit_index, sbox_type, k0)
            H_matrix[n] = leakage_model_i
        
        # CPA attack
        R_matrix = ascon_cpa(partial_traces, H_matrix)
        # Find the time sample with the maximum correlation value
        corr_vs_keyguess = np.max(np.abs(R_matrix), axis=1) # shape (8,). Max value for each column (key guess) is returned
        print("Number of traces: ", len(partial_traces))
        for j in range(corr_vs_keyguess.shape[0]):
            max_corr_value = corr_vs_keyguess[j]
            print(f"Key guess {j}: {max_corr_value:.4f}")
        # Find the key guess with the maximum correlation value
        best_key_guess = np.argmax(corr_vs_keyguess)
        print(f"Key guess with maximum correlation value: {best_key_guess}")
        print(f"Attacked bit index: {bit_index}, State register index: {state_register_index}")
        print(f"Expected key bits (bit-j, bit-j + 19, bit-j + 28): ({key_0_j}, {key_0_j19}, {key_0_j28}), got: ({(best_key_guess >> 2) & 1}, {(best_key_guess >> 1) & 1}, {(best_key_guess >> 0) & 1})")
        # Store the correlation values for all the key guesses
        corr_vs_traces.append(corr_vs_keyguess)
    # Correlation vs traces plot
    corr_vs_traces = np.array(corr_vs_traces)  # shape (steps, 8)
    x = np.arange(1, len(corr_vs_traces) + 1) * resolution
    plt.figure(figsize=(10, 5))
    for key_idx in range(8):
        plt.plot(x, corr_vs_traces[:, key_idx], label=f"Key guess {key_idx}")
    plt.xlabel("Number of traces")
    plt.ylabel("Correlation value")
    plt.title("Correlation vs Number of traces - Bit {} - S-Box {}".format(bit_index, sbox_type))
    plt.grid()
    plt.legend()
    if (save_plot):
        plt.savefig("../../../x-heep/Graphs/ASCON_c/ASCON_correlation_vs_traces" + f"_sbox_{sbox_type}_bit_{bit_index}.png")
    plt.close()
    toc = time.perf_counter()
    print(f"\nCPA attack on bit ?? completed in {(toc - tic)/60:.2f} minutes.\n")